# Resonus Room: example

Room-acoustic measurement and analysis with the same core API as the CLI and the GUI:

1. measure an impulse response on the **simulated room** (exponential sweep, resonus-core),
2. analyse it per ISO 3382-1/-2 (EDT, T20, T30, fit quality, C50, C80, D50, Ts),
3. evaluate an ISO 354 absorption measurement from synthetic reverberation-room IRs.
4. compute the IACC of an ear pair per ISO 3382-1:2009 Annex B (informative).

Status: band filters are Butterworth; IEC 61260-1 compliance is **not verified**
(PROVISIONAL, see `docs/DECISIONS.md`). Results are calculations, not those of a
type-approved instrument.

In [ ]:
import numpy as np
from resonus_core.plot import MplPlotView

from resonus_room import plotting
from resonus_room.core.engine import analyze_ir, measure, open_device
from resonus_room.core.settings import AppSettings

## 1. Level test before measuring

Plays 1 s of white noise with its peak at `settings.level` (dB re full scale) on each
output in turn and reports peak, RMS and clipping per output/input pair (GUI: Test
button with live meters; CLI: `resonus-room test-levels`; D-21). `on_block`
receives the output label and the levels of each recorded block while the noise plays.
The simulated room has unit energy (D-21), so the levels are those of a room that
neither amplifies nor attenuates.

In [ ]:
from resonus_room.core.levels import level_test, meter_zone, pair_labels

test_settings = AppSettings(simulate=True, level=-20.0)
blocks = []
results = level_test(test_settings, open_device(test_settings), lambda out, lv: blocks.append(out))
for (_out, levels), labels in zip(results, pair_labels(test_settings), strict=True):
    for label, v in zip(labels, levels, strict=True):
        print(
            f"{label}: peak {v.peak_db:.1f} dB, RMS {v.rms_db:.1f} dB, "
            f"{meter_zone(v.peak_db)}{', CLIPPED' if v.clipped else ''}"
        )
print(f"{len(blocks)} live blocks")

## 2. Measurement on the simulated room (T = 0.8 s in every band)

In [ ]:
# plot_mark: text in the bottom-right corner of the plots (default "Re:sonus", empty: none)
settings = AppSettings(duration=1.0, tail=2.0, ir_length=2.0, sim_decay_time=0.8)
result = measure(settings, open_device(settings))
ir, fr = result.impulse_responses[0], result.frequency_responses[0]
print(ir.metadata.device)

In [ ]:
view = MplPlotView(brand_text=settings.plot_mark)
plotting.setup_response_axes(view)
plotting.show_response(view, fr, "tf", "Simulated room")
view

## 3. Analysis in octave bands

In [ ]:
analysis = analyze_ir(ir, settings)
print(f"{'band':>6} {'EDT':>6} {'T20':>6} {'T30':>6} {'xi20':>6} {'C %':>6} {'PNR':>6}  flags")
for label, p in zip(analysis.bands.labels, analysis.band_results, strict=True):
    print(
        f"{label:>6} {p.edt:6.2f} {p.t20:6.2f} {p.t30:6.2f} {p.fits['T20'].xi:6.1f} "
        f"{p.curvature:6.1f} {p.curve.peak_to_noise_db:6.1f}  {' '.join(p.flags)}"
    )

In [ ]:
band = analysis.band_results[3]  # 1 kHz
t0 = analysis.onset / ir.sample_rate
view = MplPlotView(brand_text=settings.plot_mark)
plotting.setup_ir_axes(view)
plotting.show_ir(view, ir, "ir", "Impulse response")
plotting.show_decay(view, band.curve, t0, "decay", "1 kHz decay curve")
plotting.show_fit(view, band.fits["T30"], t0, "fit", "T30 fit")
view

In [ ]:
view = MplPlotView(brand_text=settings.plot_mark)
plotting.setup_rt_axes(view)
for name in ("EDT", "T20", "T30"):
    plotting.show_rt(view, analysis, name, name, name)
view

## 4. ISO 354 absorption from synthetic reverberation-room IRs

Empty room with A1 = 6 m² and a 10.8 m² specimen with a known αs profile, 4 positions
each; the evaluation (T20 per position, spatial means, Eqs. (5)-(9)) should return the
profile within the reported uncertainty.

In [ ]:
import tempfile
from pathlib import Path

from resonus_core.air import power_attenuation_coefficient
from resonus_core.bands import band_set
from scipy.io import wavfile

from resonus_room.core.iso354 import load_setup, run_absorption
from resonus_room.core.simulate import exponential_decay_ir

fs, volume, area = 16000, 200.0, 10.8
bands = band_set(3, 100, 5000)
alpha_true = np.linspace(0.1, 0.9, 18)
m = power_attenuation_coefficient(bands.center, 20.0, 50.0)


def _t_of(a):
    # ISO 354 Eq. (5) solved for T (20 degC: c = 343 m/s)
    return 55.3 * volume / (343.0 * (a + 4 * volume * m))


folder = Path(tempfile.mkdtemp())
names = {}
for kind, t60 in (("empty", _t_of(6.0)), ("specimen", _t_of(6.0 + alpha_true * area))):
    names[kind] = []
    for k in range(4):
        seed = 10 * k + len(kind)
        h = exponential_decay_ir(fs, 2 * t60.max() + 0.5, t60, bands, delay=0.01, seed=seed)
        wavfile.write(folder / f"{kind}{k}.wav", fs, (0.5 * h / np.abs(h).max()).astype(np.float32))
        names[kind].append(f"{kind}{k}.wav")
(folder / "setup.toml").write_text(
    f"""volume = {volume}
specimen_area = {area}
[empty]
temperature = 20.0
relative_humidity = 50.0
files = {names["empty"]}
[specimen]
temperature = 20.0
relative_humidity = 50.0
files = {names["specimen"]}
""".replace("'", '"')
)
run = run_absorption(load_setup(folder / "setup.toml"), AppSettings())
for issue in run.result.issues:
    print("warning:", issue)

In [ ]:
view = MplPlotView(brand_text=settings.plot_mark)
plotting.setup_absorption_axes(view)
plotting.show_absorption(view, run.result, "alpha", "Evaluated")
view.set_trace("true", run.result.frequency, alpha_true, label="Set in the simulation")
view

## 5. IACC of an ear pair (ISO 3382-1:2009 Annex B)

IACC per octave band 125 Hz to 4 kHz for the intervals early (0 to 80 ms after the
common onset), late (80 ms to the end) and full, lag range |τ| < 1 ms (B.2, B.4). The
same calculation as `resonus-room binaural LEFT RIGHT`.

Synthetic ear pair: the right ear gets 0.6 × the left response plus 0.8 × an
independent one with the same decay, so both ears have the same energy and the
expected correlation is 0.6. The IACC scatters around it (it is the maximum of |IACF|
over the lags, and the early window holds only a few periods at 125 and 250 Hz).
Identical ears give 1. The just noticeable difference is 0.075 (B.4).

In [ ]:
from resonus_room.core.binaural import IACC_BAND_RANGE, binaural_parameters
from resonus_room.core.files import binaural_to_csv

fs = 48000
iacc_bands = band_set(1, *IACC_BAND_RANGE)
t60 = np.full(len(iacc_bands), 0.8)
a = exponential_decay_ir(fs, 1.5, t60, iacc_bands, delay=0.01, seed=1)
b = exponential_decay_ir(fs, 1.5, t60, iacc_bands, delay=0.01, seed=2)
left, right = a, 0.6 * a + 0.8 * b

print(binaural_to_csv(binaural_parameters(left, right, fs, iacc_bands), iacc_bands))
print("identical ears, IACC_full:", binaural_parameters(a, a, fs, iacc_bands).full)